# Test Symmetric Dirichlet Parametrization using MeshFEM's New `MeshEnergy` Class

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys, os
sys.path.append('../')
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, benchmark
import energy

import numpy as np
import time, copy
import param_utils
import helper_funcs

In [ ]:
# m = mesh.Mesh('../../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh')
# m = mesh.Mesh('../../3rdparty/MeshFEM/misc/examples/meshes/uv-100.obj')
m = mesh.Mesh('../../../Models/TableOneModels/male.off')

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
bdry_uv = helper_funcs.getBDdataOnUnitCircle(m)

In [ ]:
# Tutte Initialization
# uv.setVars(parametrization.lscm(m).ravel())
uv_init = parametrization.harmonic(m, bdry_uv)
flip_list = parametrization.getFlips(m, uv_init)
if len(flip_list) > 0:  uv_init = parametrization.harmonic(m, bdry_uv, True)

In [ ]:
uv.setVars(uv_init.ravel())

In [ ]:
e = energy.SymmetricDirichlet(2)

In [ ]:
obj_history = []
time_histroy = []
grad_norm_history = []

hessian_projected = []
hessian_shifted_amount_list = []

step_norm_history = []
directional_derivative_history = []


def customCallback(prob, cb_ind):
    it_time = time.time()
    obj_history.append(prob.energy())
    time_histroy.append(it_time)
    grad_norm_history.append(np.linalg.norm(prob.gradient()))

def customSaveUVCallback(prob, cb_ind):
    hessian_projected.append(int(prob.hessianWasProjected))
    hessian_shifted_amount_list.append(prob.lastFactorizationShiftMagnitude)
    uv_fn = 'uv_ravel_'+ 'iter_' + str(cb_ind-1)
    uv_arr = uv.getVars()
    np.savez_compressed(os.path.join(result_path, uv_fn), arr=uv_arr)
    
def customSaveStepDCallback(prob, step, directional_derivative):
    step_norm_history.append(np.linalg.norm(step))
    directional_derivative_history.append(-directional_derivative)

In [ ]:
# Construct `SymmetricDirichlet` parametrization energy and problem
param = mesh_energy.Parametrization(m, uv, e)
param.useXBasedProjection = True

In [ ]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param])
prob.setCustomIterationCallback(customCallback)
result_path = "test_save_uv"
# prob.setCustomIterationCallback(customSaveUVCallback)
prob.setCustomLineSearchBeganCallback(customSaveStepDCallback)

In [ ]:
em = MeshFEM.EmbeddedMesh(m, uv)
v = viewer.Viewer(em, wireframe=True)
# prob.setCustomIterationCallback(v.updater())
v.show()

In [ ]:
# Work around energy nullspace by adding a small shift
prob.hessianShift = 1e-8
opt = prob.optimizer()

In [ ]:
opt.options.verboseNonPosDef = True
opt.options.niter = 200
# opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
# opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()

In [ ]:
benchmark.reset()
start_time = time.time()
opt.optimize()
benchmark.report()

In [ ]:
benchmark.totalTime('Newton iterations$')

In [ ]:
benchmark.totalTime('Catamari Symbolic Factorize$')

In [ ]:
v.update()

In [ ]:
param_viewer = param_utils.ParametrizationViewer(m, uv.getVars().reshape(-1,2))
param_viewer.show()

In [ ]:
prob.energy()

In [ ]:
len(directional_derivative_history)

In [ ]:
ele_energy_list = []
for i in range(m.numElements()):
    ele_energy_list.append(param.elementEnergy(i))
e_tested = np.sum(np.sqrt(np.array(ele_energy_list)/m.elementVolumes())) / m.numElements()
print(e_tested)

In [ ]:
2 * e_tested

In [ ]:
time_arr = np.array(time_histroy) - start_time

# Plot vs TinyAD

In [ ]:
from matplotlib import pyplot as plt
import tinyad_parametrization

In [ ]:
v_opt, obj_history_ad, grad_history_ad, time_history_ad, step_norm_history_ad, dd_history_ad = tinyad_parametrization.symmdsParamTinyAD(m, uv_init, 1000, 0.01, False)

In [ ]:
plt.figure(figsize=(8, 8))
plt.plot(obj_history, label='xbased')
plt.plot(obj_history_ad, label='TinyAD')
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.legend()

In [ ]:
plt.figure(figsize=(8, 8))
plt.plot(grad_norm_history, label='xbased')
plt.plot(grad_history_ad, label='TinyAD')
plt.yscale('log')
plt.xlabel("Iteration")
plt.ylabel("Gradient Norm")
plt.legend()

In [ ]:
name_list = ['apple', 'bee', 'tree']
print(f"Name-list: {name_list}")